# Graphs — The Comprehensive Interview Masterclass

This is the densest notebook in the DSA series — graphs are the single biggest source of FAANG technical-round failures, and worth the time. Every section opens with a **visualization-first explanation** so you can see what the algorithm is doing before reading the code, followed by the implementation with **time and space complexity** spelled out.

Out of scope by design: binary search, sorting, DP, recursion patterns, backtracking — graph problems that lean on these (e.g. word ladder DP variant) are flagged but not solved here.

## Table of contents

**Part 1: Foundations (sections 1-3)**
1. What is a graph? Vocabulary, types, real-world models
2. Representations: adjacency matrix vs adjacency list — with a decision rule
3. Building graphs in Python — the snippets you reuse everywhere

**Part 2: Traversals (sections 4-6)**
4. BFS — what it does, why it works, and four flavours
5. DFS — recursive vs iterative, when to use each
6. Connected components — the canonical use of traversal

**Part 3: Cycle detection (sections 7-8)**
7. Cycle detection in **undirected** graphs (parent-tracking)
8. Cycle detection in **directed** graphs (3-color / recursion-stack)

**Part 4: Ordering (section 9)**
9. Topological sort — Kahn's BFS and DFS-based, with DAG shortest path

**Part 5: Shortest paths (sections 10-13)**
10. Unweighted graphs — BFS
11. Non-negative weights — Dijkstra (with heap)
12. 0-1 BFS — the deque trick
13. Negative weights — Bellman-Ford; All-pairs — Floyd-Warshall

**Part 6: Spanning trees (section 14)**
14. Prim's and Kruskal's MST

**Part 7: Union-Find (section 15)**
15. Disjoint Set Union — the killer data structure

**Part 8: Advanced (section 16)**
16. SCC (Kosaraju, Tarjan), bridges, articulation points

**Part 9: Grids (section 17)**
17. Grid-as-graph problems — islands, flood fill, multi-source BFS, BFS on matrix

**Part 10: Synthesis (sections 18-19)**
18. Pattern-recognition cheat sheet
19. Interview narration & common bugs

## Reading style
Each algorithm gets four things:
1. **A picture / story** — what is the algorithm actually doing?
2. **The invariant** — the property the algorithm maintains, which is *why* it works
3. **Code with inline comments**
4. **Complexity + when to use it**


---
# 1. What is a graph?

A graph is a pair `G = (V, E)` where `V` is a set of **vertices (nodes)** and `E` is a set of **edges** between them. That is the entire definition. Everything else is vocabulary.

## Trees vs graphs vs lists
- A **list** is a graph where every node (except two) has exactly two neighbours.
- A **tree** is a graph that is connected and has no cycles. A tree with `V` nodes has exactly `V-1` edges.
- A **graph** is everything else — it can have cycles, multiple components, weights, directions.

## Types of graphs (and the questions you ask before solving any problem)

| Property | Options |
|---|---|
| **Direction** | Undirected (friends) vs directed (Twitter follows) |
| **Weights** | Unweighted vs weighted (positive only, or negative allowed) |
| **Cycles** | Cyclic vs acyclic (DAG = Directed Acyclic Graph) |
| **Connectivity** | Connected (one big chunk) vs disconnected (multiple components) |
| **Self-loops / multi-edges** | Simple graph vs multigraph |
| **Density** | Sparse (E ≈ V) vs dense (E ≈ V²) |

**ALWAYS ask these questions at the start of an interview.** The answers change which algorithm you reach for. "Can the graph have negative edges?" decides between Dijkstra and Bellman-Ford. "Is it a DAG?" decides whether topological sort is even possible.

## Degree
- **Undirected graph:** degree of a node = number of edges connected to it. Sum of all degrees = 2E (each edge contributes 2).
- **Directed graph:** in-degree = incoming edges; out-degree = outgoing. Sum of in-degrees = sum of out-degrees = E.

## Walks, paths, cycles
- **Walk:** sequence of vertices where consecutive ones are connected by an edge. Vertices and edges may repeat.
- **Path:** walk with no repeated vertices.
- **Cycle:** path that starts and ends at the same vertex (length ≥ 3 in a simple undirected graph).

## Max edges
- Undirected, simple graph: at most `V*(V-1)/2` edges (every pair connected).
- Directed, simple graph: at most `V*(V-1)` edges (each pair can have 2 edges, one each direction).


---
# 2. Representations — adjacency matrix vs adjacency list

Two ways to store a graph in memory. Picking the wrong one can change the complexity of your algorithm by a factor of V. You need to be able to justify the choice in an interview.

## 2a. Adjacency matrix
A 2D array `M` of size V×V. `M[u][v] = 1` if there's an edge from u to v, else 0. For weighted graphs, store the weight instead of 1.

```
   0  1  2  3
0 [0, 1, 1, 0]
1 [1, 0, 1, 0]
2 [1, 1, 0, 1]
3 [0, 0, 1, 0]
```

- **Space:** O(V²) — independent of how many edges actually exist
- **Check if edge exists between u, v:** O(1) — just look up `M[u][v]`
- **Add / remove edge:** O(1)
- **Find all neighbours of u:** O(V) — scan a full row
- **Symmetric** for undirected graphs

## 2b. Adjacency list
A list of lists (or a dict of lists). `adj[u]` contains the neighbours of u.

```
adj[0] = [1, 2]
adj[1] = [0, 2]
adj[2] = [0, 1, 3]
adj[3] = [2]
```

- **Space:** O(V + E) — only stores edges that exist; undirected uses 2E slots (each edge twice), directed uses E slots
- **Check if edge exists between u, v:** O(degree of u) — have to scan the list
- **Add edge:** O(1)
- **Remove edge:** O(degree)
- **Find all neighbours of u:** O(degree) — perfect

## 2c. The decision rule
- **Dense graph (E ≈ V²)** OR **frequent edge-existence queries** → adjacency matrix
- **Sparse graph (E ≪ V²)** OR **traversal-heavy** → adjacency list (this is 95% of interview problems)
- **Weighted, need fast neighbour iteration with weights** → adjacency list of tuples `[(neighbour, weight), ...]`

For interviews, default to adjacency list unless the problem screams matrix.

## A practical note
For interview problems the input is often given as `edges = [[u, v], ...]` and you build whichever representation you want. Always build the representation first — don't try to traverse the raw edge list directly, it'll cost you.


---
# 3. Building graphs — the snippets you reuse everywhere

You'll write these helper functions in every graph interview. Memorize them.

In [ ]:
# 3.1 Build adjacency list — unweighted
from collections import defaultdict, deque
from typing import List

def build_adj_list(n: int, edges: List[List[int]], directed: bool = False):
    '''
    n: number of vertices (assumed 0..n-1)
    edges: list of [u, v] pairs
    Returns: list of lists
    '''
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        if not directed:
            adj[v].append(u)
    return adj

# Sanity check
g = build_adj_list(5, [[0,1],[0,2],[1,2],[2,3],[3,4]], directed=False)
for i, nbrs in enumerate(g):
    print(f"{i}: {nbrs}")


In [ ]:
# 3.2 Build adjacency list — weighted
def build_adj_list_weighted(n: int, edges: List[List[int]], directed: bool = False):
    '''
    edges: list of [u, v, w] triples
    Returns: list of lists of (neighbour, weight)
    '''
    adj = [[] for _ in range(n)]
    for u, v, w in edges:
        adj[u].append((v, w))
        if not directed:
            adj[v].append((u, w))
    return adj

g = build_adj_list_weighted(4, [[0,1,5],[0,2,8],[1,2,10],[1,3,15],[2,3,20]], directed=False)
for i, nbrs in enumerate(g):
    print(f"{i}: {nbrs}")


In [ ]:
# 3.3 Build adjacency matrix — unweighted
def build_adj_matrix(n: int, edges: List[List[int]], directed: bool = False):
    M = [[0]*n for _ in range(n)]
    for u, v in edges:
        M[u][v] = 1
        if not directed:
            M[v][u] = 1
    return M

M = build_adj_matrix(4, [[0,1],[0,2],[1,2],[2,3]], directed=False)
for row in M:
    print(row)


In [ ]:
# 3.4 Defaultdict-based adjacency list — when vertices aren't 0..n-1 (e.g., named nodes)
def build_adj_dict(edges, directed=False):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
        if not directed:
            adj[v].append(u)
    return adj

g = build_adj_dict([("A","B"),("A","C"),("B","D")])
print(dict(g))


---
# 4. Breadth-First Search (BFS)

## The picture

Imagine you're standing on a node and you want to visit every other node. BFS visits nodes **level by level** — first your immediate neighbours (level 1), then their neighbours (level 2), then theirs (level 3), and so on. It's like ripples spreading on water.

```
Level 0:        [0]
                / \
Level 1:      [1] [2]
              /     \
Level 2:    [3]    [4]
```

You visit 0, then 1 and 2, then 3 and 4. Never further before exhausting closer.

## The invariant
When BFS pops a node from the queue, **that node is at the minimum distance from the source** (in number of edges). This is why BFS solves shortest-path on unweighted graphs — the first time you reach any node, you've found the shortest way to it.

## How it works
1. Put the source node in a **queue** and mark it visited.
2. Repeat: pop a node, look at all its neighbours. Any unvisited neighbour gets queued and marked visited.
3. Stop when the queue is empty.

**Why a queue (FIFO)?** Because we want to process all level-1 nodes before any level-2 node. A FIFO ensures we drain the current level before moving on.

## Why mark visited *before* pushing, not after popping?
A common bug: marking visited when you pop. If two level-1 nodes both push the same level-2 node, it ends up in the queue twice and gets processed twice. Mark visited the moment you enqueue.

## Complexity
- **Time:** O(V + E) — each vertex is dequeued once, and we look at each edge once (twice in undirected) over the whole traversal
- **Space:** O(V) for the visited set and the queue

## A critical Python performance note
**`list.pop(0)` is O(n).** Using a list as a queue makes BFS O(V²). Use `collections.deque` whose `popleft()` is O(1).

In [ ]:
# 4.1 BFS — from a single source, returning traversal order
def bfs(start, adj, n):
    '''Return BFS order starting from `start`. n = number of vertices.'''
    visited = [False] * n
    order = []
    q = deque([start])
    visited[start] = True
    while q:
        u = q.popleft()
        order.append(u)
        for v in adj[u]:
            if not visited[v]:
                visited[v] = True       # mark on enqueue, not on dequeue
                q.append(v)
    return order

adj = build_adj_list(5, [[0,1],[0,2],[1,2],[2,3],[3,4]])
print(bfs(0, adj, 5))                   # [0, 1, 2, 3, 4]


In [ ]:
# 4.2 BFS on a disconnected graph — visit every component
def bfs_full(adj, n):
    visited = [False] * n
    order = []
    for start in range(n):
        if visited[start]:
            continue
        # BFS from this component's start
        q = deque([start])
        visited[start] = True
        while q:
            u = q.popleft()
            order.append(u)
            for v in adj[u]:
                if not visited[v]:
                    visited[v] = True
                    q.append(v)
    return order

# Two disconnected components: {0,1,2} and {3,4}
adj = build_adj_list(5, [[0,1],[0,2],[3,4]])
print(bfs_full(adj, 5))                 # [0, 1, 2, 3, 4]


In [ ]:
# 4.3 BFS for shortest distances (unweighted) — distance from a source
def bfs_distances(start, adj, n):
    '''Returns dist[i] = shortest number of edges from `start` to i, or inf.'''
    INF = float('inf')
    dist = [INF] * n
    dist[start] = 0
    q = deque([start])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if dist[v] == INF:           # 'unvisited' is encoded as 'distance still infinity'
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

adj = build_adj_list(5, [[0,1],[0,2],[1,2],[2,3],[3,4]])
print(bfs_distances(0, adj, 5))         # [0, 1, 1, 2, 3]


In [ ]:
# 4.4 BFS that also reconstructs the shortest path
def bfs_shortest_path(start, target, adj, n):
    '''Returns the shortest path from start to target (list of vertices), or None.'''
    if start == target:
        return [start]
    parent = [-1] * n
    parent[start] = start            # sentinel — we'll never overwrite this
    q = deque([start])
    found = False
    while q and not found:
        u = q.popleft()
        for v in adj[u]:
            if parent[v] == -1:
                parent[v] = u
                if v == target:
                    found = True
                    break
                q.append(v)
    if parent[target] == -1:
        return None
    # reconstruct by walking parents backward
    path = []
    cur = target
    while cur != start:
        path.append(cur)
        cur = parent[cur]
    path.append(start)
    return path[::-1]

adj = build_adj_list(6, [[0,1],[0,2],[1,3],[2,3],[3,4],[4,5]])
print(bfs_shortest_path(0, 5, adj, 6))  # one valid path: [0,1,3,4,5] or [0,2,3,4,5]


**The four flavours of BFS to recognize**

| Flavour | Use case | Key tweak |
|---|---|---|
| Single-source traversal | Visit reachable nodes | Just a `visited` set + queue |
| Single-source shortest distance | Unweighted shortest path | Maintain `dist[]` array |
| Single-source path reconstruction | "What's the actual path?" | Maintain `parent[]` array |
| Multi-source BFS | Spread from many starts simultaneously (e.g., rotting oranges) | Push ALL sources into queue at start, with distance 0 |

Multi-source BFS is its own pattern and gets a worked example in section 17.

**Practice — BFS**

| Concept | LeetCode |
|---|---|
| Number of islands (BFS variant) | LC 200 |
| 01 Matrix (multi-source BFS) | LC 542 |
| Rotting oranges (multi-source BFS) | LC 994 |
| Shortest path in binary matrix | LC 1091 |
| Word ladder (BFS on word graph) | LC 127 |
| Open the lock | LC 752 |
| Snakes and ladders | LC 909 |


---
# 5. Depth-First Search (DFS)

## The picture

DFS goes **as deep as possible** before backtracking. Imagine walking through a maze: you commit to a path, follow it until you hit a dead end, then back up and try another. BFS spreads out evenly; DFS shoots down one branch first.

```
        0
       / \
      1   2
     /     \
    3       4
```

DFS from 0 might visit: 0 → 1 → 3 → (backtrack) → 2 → 4.

## When to use DFS over BFS

| Task | Prefer |
|---|---|
| Shortest path (unweighted) | **BFS** |
| Just need to know reachability / connected components | Either; DFS is shorter to code |
| Detect a cycle | **DFS** (or Kahn's) |
| Topological sort | **DFS** (or Kahn's) |
| Find all paths from source to target | **DFS** (with backtracking) |
| Strongly connected components | **DFS** (Kosaraju / Tarjan) |
| Memory matters and tree is deep | BFS (DFS can stack-overflow) |

## Recursive vs iterative

DFS is naturally recursive. The iterative form uses an explicit stack — same algorithm. For deep graphs in Python (recursion limit ~1000), iterative is safer; for clarity, recursive is shorter. Both are O(V + E).

## A common bug

In recursive DFS, **mark visited at the start of the call**, not the end. Otherwise you may push the same node onto the recursion stack multiple times.

## Complexity
- **Time:** O(V + E)
- **Space:** O(V) — visited set + recursion stack depth (up to V in a long chain)

In [ ]:
# 5.1 DFS — recursive (the natural form)
def dfs_recursive(start, adj, n):
    visited = [False] * n
    order = []
    def go(u):
        visited[u] = True               # mark on entry
        order.append(u)
        for v in adj[u]:
            if not visited[v]:
                go(v)
    go(start)
    return order

adj = build_adj_list(5, [[0,1],[0,2],[1,2],[2,3],[3,4]])
print(dfs_recursive(0, adj, 5))        # one valid order: [0, 1, 2, 3, 4]


In [ ]:
# 5.2 DFS — iterative (use this when recursion depth is a concern)
def dfs_iterative(start, adj, n):
    visited = [False] * n
    order = []
    stack = [start]
    while stack:
        u = stack.pop()
        if visited[u]:                  # safety check — node may have been pushed twice
            continue
        visited[u] = True
        order.append(u)
        # NOTE: order differs from recursive because we reverse-iterate
        for v in adj[u]:
            if not visited[v]:
                stack.append(v)
    return order

print(dfs_iterative(0, adj, 5))


In [ ]:
# 5.3 DFS on disconnected graph — visit every component
def dfs_full(adj, n):
    visited = [False] * n
    order = []
    def go(u):
        visited[u] = True
        order.append(u)
        for v in adj[u]:
            if not visited[v]:
                go(v)
    for start in range(n):
        if not visited[start]:
            go(start)
    return order

# Two components: {0,1,2} and {3,4}
adj = build_adj_list(5, [[0,1],[0,2],[3,4]])
print(dfs_full(adj, 5))                # [0, 1, 2, 3, 4]


**Why does the iterative DFS need `if visited[u]: continue` at the top?**

In recursive DFS, we check `if not visited[v]` BEFORE recursing, so we never enter a visited node. In iterative DFS, we push neighbours **before** processing them — so by the time we pop one, another sibling may have visited it. The `if visited` check handles this.

**Practice — DFS**

| Concept | LeetCode |
|---|---|
| Number of islands (DFS) | LC 200 |
| Max area of island | LC 695 |
| Flood fill | LC 733 |
| Surrounded regions | LC 130 |
| Pacific Atlantic water flow | LC 417 |
| Clone graph | LC 133 |
| All paths from source to target (DAG) | LC 797 |


---
# 6. Connected components — your first real graph problem

## The picture
A graph might be one big connected blob, or it might be several disconnected blobs (components). A "connected component" is a maximal set of vertices that can all reach each other.

```
[0]---[1]      [3]---[4]
  \   /
   [2]                          [5]
```

Three components: `{0,1,2}`, `{3,4}`, `{5}`.

## The algorithm
For every unvisited vertex, run BFS or DFS from it. Each launch finds one entire component. Increment a counter or collect the component nodes.

## Complexity
O(V + E) — every vertex and every edge is touched once across the entire procedure.

## Why it's a building block
Many problems ask you to "count components" or "process each component separately." Recognize the pattern: **outer loop over all vertices + inner BFS/DFS guarded by a visited check.**

In [ ]:
# 6.1 Count connected components — LC 323 / LC 547 variants
def count_components(n, edges):
    adj = build_adj_list(n, edges)
    visited = [False] * n
    count = 0
    for start in range(n):
        if visited[start]:
            continue
        count += 1
        # BFS this component
        q = deque([start])
        visited[start] = True
        while q:
            u = q.popleft()
            for v in adj[u]:
                if not visited[v]:
                    visited[v] = True
                    q.append(v)
    return count

assert count_components(5, [[0,1],[1,2],[3,4]]) == 2
assert count_components(5, [[0,1],[1,2],[2,3],[3,4]]) == 1
assert count_components(5, []) == 5
print("count_components OK")


In [ ]:
# 6.2 Return the components themselves (list of vertex lists)
def get_components(n, edges):
    adj = build_adj_list(n, edges)
    visited = [False] * n
    components = []
    for start in range(n):
        if visited[start]:
            continue
        comp = []
        q = deque([start])
        visited[start] = True
        while q:
            u = q.popleft()
            comp.append(u)
            for v in adj[u]:
                if not visited[v]:
                    visited[v] = True
                    q.append(v)
        components.append(comp)
    return components

print(get_components(5, [[0,1],[1,2],[3,4]]))   # [[0,1,2],[3,4]]


**Practice — connected components**

| Concept | LeetCode |
|---|---|
| Number of connected components in undirected graph | LC 323 |
| Number of provinces (matrix variant) | LC 547 |
| Friend circles | LC 547 |
| Number of islands (grid variant — see §17) | LC 200 |
| Graph valid tree (component + cycle check) | LC 261 |


---
# 7. Cycle detection — undirected graph

## The picture
A cycle is a path that returns to its starting vertex. In an undirected graph, the simplest cycle is a triangle `A-B-C-A`. We detect it during DFS.

## The trap (and why undirected is different from directed)
Consider edge `0—1`. During DFS from 0, we visit 1. Then from 1, we look at its neighbours and see... 0, which is already visited! Naively, you'd say "we found a back edge, cycle!" But that's wrong — 0 is just the parent we came from. Every undirected edge looks like a "cycle" if you don't track who you came from.

## The fix
Pass the parent into the recursion. A neighbour that's visited **but not the parent** is a real cycle.

## The algorithm
DFS from each unvisited vertex. For each neighbour:
- If unvisited → recurse with current node as parent
- If visited AND not the parent → cycle detected

## Complexity
O(V + E) time, O(V) space.

In [ ]:
# 7.1 Cycle detection in undirected graph — DFS with parent tracking
def has_cycle_undirected(n, edges):
    adj = build_adj_list(n, edges)
    visited = [False] * n
    def dfs(u, parent):
        visited[u] = True
        for v in adj[u]:
            if not visited[v]:
                if dfs(v, u):
                    return True
            elif v != parent:           # visited AND not the one we came from -> real cycle
                return True
        return False
    for start in range(n):
        if not visited[start]:
            if dfs(start, -1):
                return True
    return False

assert has_cycle_undirected(5, [[0,1],[0,2],[1,2],[2,3],[3,4]]) == True   # triangle 0-1-2
assert has_cycle_undirected(5, [[0,1],[0,2],[2,3],[3,4]]) == False         # tree
print("has_cycle_undirected OK")


In [ ]:
# 7.2 Alternative: BFS with parent tracking — same logic, no recursion
def has_cycle_undirected_bfs(n, edges):
    adj = build_adj_list(n, edges)
    visited = [False] * n
    for start in range(n):
        if visited[start]:
            continue
        q = deque([(start, -1)])         # (node, parent)
        visited[start] = True
        while q:
            u, parent = q.popleft()
            for v in adj[u]:
                if not visited[v]:
                    visited[v] = True
                    q.append((v, u))
                elif v != parent:
                    return True
    return False

assert has_cycle_undirected_bfs(5, [[0,1],[0,2],[1,2],[2,3],[3,4]]) == True
assert has_cycle_undirected_bfs(4, [[0,1],[1,2],[2,3]]) == False
print("has_cycle_undirected_bfs OK")


**Alternative — Union-Find for undirected cycle detection (covered in §15):**
For each edge `(u, v)`, if `u` and `v` are already in the same connected component (same root), this edge creates a cycle. Cleaner and often preferred when you're already doing connectivity work.

**Practice — undirected cycle detection**

| Concept | LeetCode |
|---|---|
| Graph valid tree | LC 261 (V-1 edges + connected + no cycle, or use Union-Find) |
| Redundant connection | LC 684 (Union-Find shines here) |
| Number of provinces | LC 547 |


---
# 8. Cycle detection — directed graph (THE classic interview trap)

## Why directed is different
In a directed graph, the parent-tracking trick **doesn't work**. Why? Consider edges `0→1` and `1→0` — that's a cycle, but in DFS from 0, when you visit 1 and look at its neighbours, you'd see 0 (visited, was the parent) and incorrectly conclude no cycle.

The right insight: in a directed graph, a cycle exists iff DFS encounters a **back edge** — an edge to a vertex that is currently **on the recursion stack** (an ancestor in the DFS tree).

## Three-color method (intuitive)

Each vertex is one of:
- **WHITE** = unvisited
- **GRAY** = currently being explored (on the recursion stack right now)
- **BLACK** = fully explored (DFS from this node has finished)

DFS rule: if you reach a GRAY neighbour, you've found a cycle. (BLACK means "already done, no cycle here that we'd be in.")

## Equivalent: `recursion_stack` array

Maintain a boolean `on_stack[]`. Set to True when entering DFS for a vertex; False when leaving. If you find a neighbour with `on_stack[v] == True`, cycle!

This is exactly the same algorithm, just with a different variable name.

## Why these *two* arrays?
- `visited[]` — "have I ever started exploring this node?" Lets us skip work on revisits.
- `on_stack[]` — "is this node currently in the DFS stack above me?" Distinguishes back edges (cycle) from cross/forward edges (no cycle).

You need both. With only `visited[]`, you'd flag every revisit as a cycle even though many revisits are perfectly legal.

## Complexity
O(V + E) time, O(V) space.

In [ ]:
# 8.1 Directed cycle detection — recursion stack / 3-color
def has_cycle_directed(n, edges):
    adj = build_adj_list(n, edges, directed=True)
    visited = [False] * n
    on_stack = [False] * n
    def dfs(u):
        visited[u] = True
        on_stack[u] = True
        for v in adj[u]:
            if not visited[v]:
                if dfs(v):
                    return True
            elif on_stack[v]:           # back edge — v is an ancestor in current DFS tree
                return True
        on_stack[u] = False             # done exploring — pop from stack
        return False
    for start in range(n):
        if not visited[start]:
            if dfs(start):
                return True
    return False

assert has_cycle_directed(5, [[0,1],[0,2],[1,2],[2,3],[3,4]]) == False
assert has_cycle_directed(5, [[0,1],[1,2],[2,3],[3,4],[4,1]]) == True   # 1->2->3->4->1
assert has_cycle_directed(3, [[0,1],[1,2],[2,0]]) == True               # 0->1->2->0
print("has_cycle_directed OK")


In [ ]:
# 8.2 Cycle detection via topological sort — Kahn's algorithm
# IDEA: if you can produce a full topological ordering, no cycle exists.
# If not all vertices end up in the topo order, the leftover ones are in a cycle.
def has_cycle_directed_kahn(n, edges):
    adj = build_adj_list(n, edges, directed=True)
    in_deg = [0] * n
    for u in range(n):
        for v in adj[u]:
            in_deg[v] += 1
    q = deque(i for i in range(n) if in_deg[i] == 0)
    processed = 0
    while q:
        u = q.popleft()
        processed += 1
        for v in adj[u]:
            in_deg[v] -= 1
            if in_deg[v] == 0:
                q.append(v)
    return processed != n               # didn't finish -> cycle remains

assert has_cycle_directed_kahn(5, [[0,1],[0,2],[1,2],[2,3],[3,4]]) == False
assert has_cycle_directed_kahn(5, [[0,1],[1,2],[2,3],[3,4],[4,1]]) == True
print("has_cycle_directed_kahn OK")


**The two ways are equivalent in time complexity but feel different:**
- **DFS / 3-color** — natural for "find the cycle" problems, or when you'll do other DFS work anyway
- **Kahn's BFS** — natural for "find any valid ordering" + cycle detection as a byproduct

**Practice — directed cycle detection**

| Concept | LeetCode |
|---|---|
| Course schedule (can you finish?) | LC 207 |
| Course schedule II (return order) | LC 210 |
| Find eventual safe states | LC 802 |
| Find the celebrity | LC 277 |
| Loud and rich | LC 851 |


---
# 9. Topological sort

## The picture
You have a list of tasks with dependencies. Course A is a prerequisite for Course B. Cooking step "boil water" must come before "add pasta". You want a **linear ordering** of tasks such that every prerequisite comes before the things that depend on it.

A topological sort exists **if and only if the graph is a DAG** (Directed Acyclic Graph). Cycles mean impossible dependencies.

## Two algorithms

### 9a. Kahn's algorithm (BFS-based) — recommended

**Idea:** repeatedly pick a vertex with no prerequisites (in-degree 0), output it, and "remove" it from the graph by decrementing the in-degrees of its successors. New zero-in-degree vertices become available; queue them up.

**Why it works:** by definition, an in-degree-0 vertex has no prerequisites, so it can come first. Once you take it, its outgoing edges no longer count, which might unblock the next layer. Continue until empty.

**Cycle detection bonus:** if at the end you haven't processed all `n` vertices, you have a cycle.

### 9b. DFS-based

**Idea:** do DFS. The moment you finish processing a vertex (return from its DFS call), push it onto a stack. At the end, pop the stack to get the topological order.

**Why it works:** when DFS finishes a vertex, every node reachable from it has already finished — so they should all come AFTER it in the order. The stack reverses this so they come after when we pop.

## Complexity (both)
O(V + E) time, O(V) space.

In [ ]:
# 9a. Kahn's algorithm — BFS-based topological sort
def topo_sort_kahn(n, edges):
    adj = build_adj_list(n, edges, directed=True)
    in_deg = [0] * n
    for u in range(n):
        for v in adj[u]:
            in_deg[v] += 1
    q = deque(i for i in range(n) if in_deg[i] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in adj[u]:
            in_deg[v] -= 1
            if in_deg[v] == 0:
                q.append(v)
    if len(order) != n:
        return []                       # cycle detected -> no valid topo sort
    return order

print(topo_sort_kahn(5, [[0,2],[0,3],[1,3],[1,4]]))     # one valid: [0,1,2,3,4]
print(topo_sort_kahn(5, [[0,1],[1,2],[2,0]]))           # [] — cycle


In [ ]:
# 9b. DFS-based topological sort
def topo_sort_dfs(n, edges):
    adj = build_adj_list(n, edges, directed=True)
    visited = [False] * n
    on_stack = [False] * n
    order = []                          # finish order
    cycle_found = [False]
    def dfs(u):
        visited[u] = True
        on_stack[u] = True
        for v in adj[u]:
            if not visited[v]:
                dfs(v)
                if cycle_found[0]: return
            elif on_stack[v]:
                cycle_found[0] = True
                return
        on_stack[u] = False
        order.append(u)                 # FINISH — append after children done
    for s in range(n):
        if not visited[s]:
            dfs(s)
            if cycle_found[0]:
                return []
    return order[::-1]                  # reverse the finish order

print(topo_sort_dfs(5, [[0,1],[2,3],[1,3],[2,4],[3,4]]))


In [ ]:
# 9c. Shortest path in a DAG — topological sort + relax
# WORKS for DAGs (even with negative weights). For general weighted graphs, use Dijkstra or Bellman-Ford.
def shortest_path_dag(n, weighted_edges, source):
    '''weighted_edges = list of [u, v, w]. Returns dist[] from source.'''
    adj = build_adj_list_weighted(n, weighted_edges, directed=True)
    # We need a topo sort — strip weights first
    raw_edges = [[u, v] for u, v, _ in weighted_edges]
    order = topo_sort_kahn(n, raw_edges)
    INF = float('inf')
    dist = [INF] * n
    dist[source] = 0
    for u in order:
        if dist[u] == INF:
            continue                    # unreachable from source
        for v, w in adj[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    return dist

# 0 -[5]-> 1 -[3]-> 2 -[7]-> 3
#  \________[10]____________/
edges_w = [[0,1,5],[1,2,3],[2,3,7],[0,3,10]]
print(shortest_path_dag(4, edges_w, 0))   # [0, 5, 8, 10]


**Why DAG shortest path is easier than general:** with no cycles, once we process a vertex in topological order, all paths *to* it have already been considered. No need for the priority queue / repeated relaxation Dijkstra and Bellman-Ford require.

**Practice — topological sort**

| Concept | LeetCode |
|---|---|
| Course schedule | LC 207 |
| Course schedule II | LC 210 |
| Alien dictionary | LC 269 (hard, classic) |
| Sequence reconstruction | LC 444 |
| Parallel courses (longest-path on DAG) | LC 1136 |
| Shortest path in DAG | LC 1976 variant |
| Find all possible recipes | LC 2115 |


---
# 10. Shortest path — the family

A quick map of which algorithm to use:

| Graph type | Algorithm | Time |
|---|---|---|
| Unweighted (or all weights equal) | **BFS** | O(V + E) |
| DAG (any weights, including negative) | **DAG shortest path** (toposort + relax) | O(V + E) |
| Weights only 0 or 1 | **0-1 BFS** (deque) | O(V + E) |
| Non-negative weights | **Dijkstra** (with heap) | O((V+E) log V) |
| Negative weights allowed (no negative cycles) | **Bellman-Ford** | O(V · E) |
| All-pairs shortest paths (dense) | **Floyd-Warshall** | O(V³) |
| With heuristic (e.g., grid) | **A*** | depends on heuristic |

The questions to ask: *is it weighted? are weights positive? is it a DAG? do I need all-pairs or one-source?*

---
# 11. Dijkstra's algorithm

## The picture
Imagine the graph is a road network. You're at node 0 and want the shortest driving distance to every other city. Dijkstra always picks the **closest unfinalized city** next, finalizes it, and uses it to relax (improve) the distances to its neighbours. Like ripples that travel slower over longer edges.

## The invariant
Once a vertex is popped from the priority queue, its recorded distance is the true shortest distance. We never need to revisit it.

## Why this needs non-negative weights
The "closest unfinalized city is final" argument assumes the path to it can't get shorter through some longer detour. With negative edges, a longer-looking path could suddenly become cheap, breaking the greedy choice.

## Two implementations

### Naive O(V²) — adjacency matrix
At each step, scan all V vertices to find the unfinalized one with smallest distance. Good for **dense graphs** where E ≈ V².

### Heap-based O((V + E) log V) — adjacency list
Use a min-heap to find the minimum in O(log V). For sparse graphs (E ≪ V²), this is faster.

## A common bug — "stale" heap entries

When we update `dist[v]`, we push `(new_dist, v)` to the heap. The OLD entry for `v` may still be sitting in the heap with a worse distance. Solution: when popping, if the popped distance doesn't match `dist[v]`, skip — it's stale.

Alternative: use a decrease-key heap (Python doesn't have a great one built in; the "lazy" approach above is standard).

In [ ]:
# 11.1 Dijkstra — heap-based, the version you should default to
import heapq

def dijkstra(n, weighted_edges, source, directed=False):
    adj = build_adj_list_weighted(n, weighted_edges, directed=directed)
    INF = float('inf')
    dist = [INF] * n
    dist[source] = 0
    # heap entries: (current_known_distance, vertex)
    pq = [(0, source)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue                    # stale — we've already found a better path
        for v, w in adj[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

edges_w = [[0,1,4],[0,2,1],[2,1,2],[1,3,1],[2,3,5]]
print(dijkstra(4, edges_w, 0))         # [0, 3, 1, 4]   — 0->2->1 cheaper than direct 0->1


In [ ]:
# 11.2 Dijkstra with path reconstruction
def dijkstra_with_path(n, weighted_edges, source, target, directed=False):
    adj = build_adj_list_weighted(n, weighted_edges, directed=directed)
    INF = float('inf')
    dist = [INF] * n
    parent = [-1] * n
    dist[source] = 0
    pq = [(0, source)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        if u == target:
            break                       # optional optimization — early exit
        for v, w in adj[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                parent[v] = u
                heapq.heappush(pq, (nd, v))
    if dist[target] == INF:
        return INF, []
    # reconstruct
    path = []
    cur = target
    while cur != -1:
        path.append(cur)
        cur = parent[cur]
    return dist[target], path[::-1]

print(dijkstra_with_path(4, edges_w, 0, 3))    # (4, [0, 2, 1, 3])


In [ ]:
# 11.3 Dijkstra — naive O(V^2) version (for very dense graphs)
def dijkstra_naive(n, weighted_edges, source, directed=False):
    M = [[0]*n for _ in range(n)]
    for u, v, w in weighted_edges:
        M[u][v] = w
        if not directed: M[v][u] = w
    INF = float('inf')
    dist = [INF] * n
    dist[source] = 0
    finalized = [False] * n
    for _ in range(n):
        # pick the closest unfinalized
        u, best = -1, INF
        for i in range(n):
            if not finalized[i] and dist[i] < best:
                u, best = i, dist[i]
        if u == -1: break
        finalized[u] = True
        for v in range(n):
            if M[u][v] != 0 and not finalized[v]:
                if dist[u] + M[u][v] < dist[v]:
                    dist[v] = dist[u] + M[u][v]
    return dist

print(dijkstra_naive(4, edges_w, 0))  # [0, 3, 1, 4]


**The two pitfalls everyone falls into**
1. **Negative edges.** Dijkstra is incorrect with negative weights. Use Bellman-Ford.
2. **Stale heap entries.** Without the `if d > dist[u]: continue` check, you'll relax neighbours of an already-better-found vertex. Doesn't break correctness but bloats the runtime.

**Practice — Dijkstra**

| Concept | LeetCode |
|---|---|
| Network delay time | LC 743 |
| Path with minimum effort | LC 1631 (Dijkstra on grid with "max edge" relaxation) |
| Cheapest flights within K stops | LC 787 (Dijkstra variant, also Bellman-Ford) |
| Path with maximum probability | LC 1514 (Dijkstra with max-heap) |
| Swim in rising water | LC 778 |
| Reachable nodes in subdivided graph | LC 882 |


---
# 12. 0-1 BFS — the deque trick

## When you'd reach for it
Edges have weights that are **only 0 or 1**. Dijkstra works but the heap overhead is unnecessary. 0-1 BFS gives you O(V + E).

## The trick
Use a `deque` instead of a queue:
- Edge of weight **0** → `appendleft` (treat as same level)
- Edge of weight **1** → `append` (treat as next level)

The deque preserves the invariant that the front always holds the smallest distance.

## Where this shows up
- Grid problems where some moves are free and some cost 1 (e.g., LC 1368)
- Shortest path with "skip" operations
- Bipartite-ish weight structures

In [ ]:
# 12.1 0-1 BFS
def zero_one_bfs(n, edges_with_01_weights, source):
    '''edges as [u, v, w] with w in {0, 1}. Directed assumed; flip the helper if needed.'''
    adj = build_adj_list_weighted(n, edges_with_01_weights, directed=True)
    INF = float('inf')
    dist = [INF] * n
    dist[source] = 0
    dq = deque([source])
    while dq:
        u = dq.popleft()
        for v, w in adj[u]:
            nd = dist[u] + w
            if nd < dist[v]:
                dist[v] = nd
                if w == 0:
                    dq.appendleft(v)
                else:
                    dq.append(v)
    return dist

edges = [[0,1,1],[0,2,0],[2,1,0],[1,3,1],[2,3,1]]
print(zero_one_bfs(4, edges, 0))  # [0, 0, 0, 1]


---
# 13. Bellman-Ford & Floyd-Warshall

## 13a. Bellman-Ford — for negative weights

### The picture
Dijkstra greedily finalizes the closest vertex; Bellman-Ford doesn't pick favourites — it just **relaxes every edge V-1 times**.

### Why V-1 iterations?
The shortest path between any two vertices has at most V-1 edges (else it would repeat a vertex, which can only happen in cycles). One iteration of "relax all edges" guarantees that all paths of length 1 are correct; two iterations cover length 2; ...; V-1 iterations cover all possible shortest paths.

### Cycle detection bonus
After V-1 iterations, any further relaxation means a **negative cycle** exists.

### Complexity
O(V · E) time, O(V) space.

In [ ]:
# 13.1 Bellman-Ford
def bellman_ford(n, weighted_edges, source, directed=True):
    '''
    Returns (dist, has_negative_cycle).
    For undirected, add each edge twice (or set directed=False and helper adds both).
    '''
    INF = float('inf')
    dist = [INF] * n
    dist[source] = 0
    edges = list(weighted_edges)
    if not directed:
        edges += [[v, u, w] for u, v, w in weighted_edges]
    # V-1 relaxation rounds
    for _ in range(n - 1):
        updated = False
        for u, v, w in edges:
            if dist[u] != INF and dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                updated = True
        if not updated:
            break                       # early termination — converged
    # one more pass to detect negative cycle
    for u, v, w in edges:
        if dist[u] != INF and dist[u] + w < dist[v]:
            return dist, True
    return dist, False

# Graph with a negative edge
edges_neg = [[0,1,4],[0,2,5],[1,2,-3],[2,3,4]]
d, neg = bellman_ford(4, edges_neg, 0)
print(d, neg)               # [0, 4, 1, 5], False


## 13b. Floyd-Warshall — all-pairs shortest paths

### The picture
For every pair `(i, j)`, find the shortest path. Brute force: run Dijkstra from every vertex → O(V · (V+E) log V). Floyd-Warshall does it in O(V³) by a dynamic-programming trick.

### The recurrence
`dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])` — "the shortest path from i to j either doesn't use k, or it does and breaks at k."

### When it wins
Dense graphs where E ≈ V². For sparse graphs, V Dijkstras is faster.

### Negative weights
Floyd-Warshall handles them. Detects negative cycles iff any `dist[i][i]` becomes negative.

In [ ]:
# 13.2 Floyd-Warshall — all-pairs shortest paths
def floyd_warshall(n, weighted_edges, directed=True):
    INF = float('inf')
    dist = [[INF]*n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
    for u, v, w in weighted_edges:
        dist[u][v] = min(dist[u][v], w)
        if not directed:
            dist[v][u] = min(dist[v][u], w)
    # the triple loop — k MUST be the outermost
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    return dist

D = floyd_warshall(4, [[0,1,5],[1,2,3],[0,2,10],[2,3,1]])
for row in D:
    print(row)


**Why k MUST be the outermost loop in Floyd-Warshall:** the recurrence assumes you've considered using intermediate vertices `{0, 1, ..., k-1}` before considering whether to use `k`. If you put `k` inside, you'd reuse stale values from the same iteration and get wrong answers. This is the single most common Floyd-Warshall bug.

**Practice — Bellman-Ford & Floyd-Warshall**

| Concept | LeetCode |
|---|---|
| Cheapest flights within K stops | LC 787 (Bellman-Ford fits perfectly: K+1 rounds) |
| Find the city with smallest number of neighbors | LC 1334 (Floyd-Warshall) |
| Detect negative cycle | (GFG classic) |


---
# 14. Minimum Spanning Tree — Prim's & Kruskal's

## The picture
You have an undirected, weighted graph (think: a network of cities with road-building costs). You want to **connect every city** while **minimizing total cost**. The result is a tree (no cycles, V-1 edges) that **spans** all V vertices with minimum total weight.

This is called a **Minimum Spanning Tree (MST)**. The MST exists iff the graph is connected.

## Why the answer is a tree
If you had a cycle, you could remove the heaviest edge in that cycle and still be connected — saving cost. Optimal = no cycles = tree.

## Two greedy algorithms — both correct, different intuitions

### 14a. Prim's — "grow the tree from a seed"
Start with any one vertex. At each step, add the **cheapest edge** that connects a vertex IN the tree to one OUTSIDE. Repeat V-1 times.

Like a tree growing — always picking the cheapest branch to extend.

### 14b. Kruskal's — "merge components"
Sort all edges by weight. Walk through them: if an edge connects two **different** components, add it; otherwise skip (would create a cycle). Stop when you have V-1 edges.

Like an archipelago becoming connected by building the cheapest bridges first.

## Decision rule
- **Dense graph** → Prim's (adjacency-matrix version is O(V²))
- **Sparse graph** → Kruskal's (sort dominates, O(E log E))
- **Already have Union-Find for other reasons** → Kruskal's

## Complexity
- Prim's (matrix-based): O(V²)
- Prim's (heap-based): O((V + E) log V) ≈ O(E log V)
- Kruskal's: O(E log E) for sorting + near-linear for Union-Find

In [ ]:
# 14a.1 Prim's — heap-based (sparse graphs)
import heapq

def prim_mst(n, weighted_edges):
    adj = build_adj_list_weighted(n, weighted_edges, directed=False)
    in_mst = [False] * n
    total = 0
    # min-heap of (weight, vertex_to_add)
    pq = [(0, 0)]                       # start with vertex 0 at cost 0
    while pq:
        w, u = heapq.heappop(pq)
        if in_mst[u]:
            continue
        in_mst[u] = True
        total += w
        for v, ew in adj[u]:
            if not in_mst[v]:
                heapq.heappush(pq, (ew, v))
    return total

assert prim_mst(4, [[0,1,5],[0,2,8],[1,2,10],[1,3,15],[2,3,20]]) == 28
print("prim_mst OK")


In [ ]:
# 14a.2 Prim's — O(V^2) version (dense graphs, no heap)
def prim_mst_dense(n, weighted_edges):
    M = [[0]*n for _ in range(n)]
    for u, v, w in weighted_edges:
        M[u][v] = w; M[v][u] = w
    INF = float('inf')
    key = [INF] * n
    in_mst = [False] * n
    key[0] = 0
    total = 0
    for _ in range(n):
        u = -1
        for i in range(n):
            if not in_mst[i] and (u == -1 or key[i] < key[u]):
                u = i
        in_mst[u] = True
        total += key[u]
        for v in range(n):
            if M[u][v] != 0 and not in_mst[v]:
                key[v] = min(key[v], M[u][v])
    return total

assert prim_mst_dense(4, [[0,1,5],[0,2,8],[1,2,10],[1,3,15],[2,3,20]]) == 28
print("prim_mst_dense OK")


(Kruskal's needs Union-Find — we'll do it in the next section.)

---
# 15. Union-Find (Disjoint Set Union — DSU)

## The picture
You have a bunch of elements. They start in their own groups (each element alone). Over time, you "merge" groups together. At any moment, you can ask "are these two elements in the same group?"

This data structure handles those queries in near-constant time. It's the foundation of Kruskal's, dynamic connectivity, account merging, and a dozen other problems.

## Two operations
- **find(x)** — returns the "representative" (root) of the group containing x
- **union(x, y)** — merges the groups containing x and y

Two elements are in the same group iff `find(x) == find(y)`.

## How it's stored
A `parent[]` array. `parent[x]` points to x's parent. Each tree's root has `parent[root] == root`. To find the root, walk up the parent chain.

```
parent = [0, 0, 1, 1, 4]     index: 0 1 2 3 4

Tree:    0          4
         |
         1
        / \
       2   3
```

## Two optimizations (you must use both)

### Path compression (in `find`)
After walking up to the root, point every node on that path directly to the root. Future lookups are O(1).

### Union by rank (or size)
When merging two trees, hang the **shorter** under the **taller** (or smaller under larger). This keeps trees shallow.

## Complexity with both optimizations
Effectively **O(α(n))** per operation, where α is the inverse Ackermann function. For all practical n (up to ~10^80), α(n) ≤ 4. So: **effectively O(1) per operation**.

Without optimizations: O(log n) per operation with one of them; O(n) per operation with neither (it degrades to a linked list).

In [ ]:
# 15.1 Union-Find — clean, idiomatic Python implementation
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))    # each starts as its own root
        self.rank   = [0] * n           # tree height upper bound
        self.size   = [1] * n           # component sizes (handy bonus)
        self.count  = n                 # number of disjoint components

    def find(self, x):
        # Path compression — iterative version (avoids recursion stack overflow)
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            nxt = self.parent[x]
            self.parent[x] = root       # flatten
            x = nxt
        return root

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False                # already in same set — useful for cycle detection!
        # Union by rank
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        self.size[rx] += self.size[ry]
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        self.count -= 1
        return True

    def connected(self, x, y):
        return self.find(x) == self.find(y)

# Quick test
uf = UnionFind(5)
uf.union(0, 1)
uf.union(2, 3)
uf.union(1, 3)
print(uf.connected(0, 3))   # True — 0-1-3-2 all merged
print(uf.connected(0, 4))   # False
print(uf.count)             # 2  ({0,1,2,3}, {4})


In [ ]:
# 15.2 Kruskal's MST — finally
def kruskal_mst(n, weighted_edges):
    edges = sorted(weighted_edges, key=lambda e: e[2])
    uf = UnionFind(n)
    total = 0
    used = 0
    for u, v, w in edges:
        if uf.union(u, v):              # union returns True only if they were different sets
            total += w
            used += 1
            if used == n - 1:
                break
    return total if used == n - 1 else -1   # -1 if graph wasn't connected

assert kruskal_mst(4, [[0,1,5],[0,2,8],[1,2,10],[1,3,15],[2,3,20]]) == 28
print("kruskal_mst OK")


In [ ]:
# 15.3 Cycle detection in undirected graph using Union-Find
# For each edge, if both endpoints already share a root, the edge would close a cycle.
def has_cycle_uf(n, edges):
    uf = UnionFind(n)
    for u, v in edges:
        if not uf.union(u, v):
            return True
    return False

assert has_cycle_uf(5, [[0,1],[0,2],[1,2],[2,3],[3,4]]) == True
assert has_cycle_uf(5, [[0,1],[0,2],[2,3],[3,4]]) == False
print("has_cycle_uf OK")


In [ ]:
# 15.4 Number of connected components via UF
def count_components_uf(n, edges):
    uf = UnionFind(n)
    for u, v in edges:
        uf.union(u, v)
    return uf.count

assert count_components_uf(5, [[0,1],[1,2],[3,4]]) == 2
print("count_components_uf OK")


**When to reach for Union-Find vs BFS/DFS for connectivity**
- **Static graph, count components once** → BFS/DFS is fine
- **Dynamic graph — edges arrive over time, you need running connectivity queries** → Union-Find is much better
- **Anything that adds edges and asks "are these merged yet?"** → Union-Find

**Practice — Union-Find**

| Concept | LeetCode |
|---|---|
| Number of provinces | LC 547 |
| Number of connected components | LC 323 |
| Graph valid tree | LC 261 |
| Redundant connection | LC 684 |
| Redundant connection II (directed, harder) | LC 685 |
| Accounts merge | LC 721 |
| Most stones removed with same row or column | LC 947 |
| Number of islands II (online) | LC 305 |
| Satisfiability of equality equations | LC 990 |
| Smallest string with swaps | LC 1202 |
| Earliest moment when everyone become friends | LC 1101 |
| Minimum number of swaps to make string balanced | (variant) |


---
# 16. Advanced — SCC, bridges, articulation points (lighter touch)

These are rare in interviews but worth knowing exist for senior rounds. We give intuition + code for the most common (Kosaraju's SCC), brief notes on the rest.

## 16a. Strongly Connected Components (SCC)

### The picture
In a **directed** graph, an SCC is a maximal group of vertices where every vertex can reach every other. Imagine cities with one-way roads where any city in the group can drive to any other (possibly through several roads).

### Why it matters
"Reduce" a directed graph by collapsing each SCC into a single node — what remains is a DAG. This is the basis of compiler analysis, dependency resolution, deadlock detection.

### Kosaraju's algorithm — two DFS passes
1. DFS on original graph. Push each vertex onto a stack when finishing.
2. Reverse all edges.
3. Pop from the stack; for each unvisited vertex, DFS in the reversed graph. Each launch finds one SCC.

### Why it works
The first DFS gives finish times in topological order of the DAG-of-SCCs. The reversed graph has the same SCCs but the "leaders" (which would have been roots in the topo order) are now sinks — so DFS from them stays within one SCC.

### Complexity
O(V + E).

In [ ]:
# 16.1 Kosaraju's SCC
def kosaraju_scc(n, edges):
    adj = build_adj_list(n, edges, directed=True)
    radj = build_adj_list(n, [[v, u] for u, v in edges], directed=True)
    visited = [False] * n
    order = []
    # Pass 1: DFS on original, record finish order
    def dfs1(u):
        stack = [(u, iter(adj[u]))]
        visited[u] = True
        while stack:
            v, it = stack[-1]
            nxt = next(it, None)
            if nxt is None:
                order.append(v)
                stack.pop()
            elif not visited[nxt]:
                visited[nxt] = True
                stack.append((nxt, iter(adj[nxt])))
    for s in range(n):
        if not visited[s]:
            dfs1(s)
    # Pass 2: DFS on REVERSED graph in reverse-finish order
    visited = [False] * n
    sccs = []
    for s in reversed(order):
        if not visited[s]:
            comp = []
            st = [s]
            visited[s] = True
            while st:
                u = st.pop()
                comp.append(u)
                for v in radj[u]:
                    if not visited[v]:
                        visited[v] = True
                        st.append(v)
            sccs.append(comp)
    return sccs

# 0 -> 1 -> 2 -> 0  (one SCC: {0,1,2})  and  3 -> 4  (two singleton SCCs: {3}, {4})
print(kosaraju_scc(5, [[0,1],[1,2],[2,0],[3,4]]))


## 16b. Bridges and articulation points (Tarjan)

### Definitions
- **Articulation point** (or "cut vertex"): a vertex whose removal disconnects the graph.
- **Bridge** (or "cut edge"): an edge whose removal disconnects the graph.

### Tarjan's algorithm (one DFS pass, O(V+E))
Track for each vertex two values:
- `disc[u]` = the time when u was first discovered
- `low[u]` = the lowest `disc` reachable from u via DFS-tree descendants and at most one back edge

**Bridge condition:** edge (u, v) is a bridge iff `low[v] > disc[u]` — meaning from v, you can't reach u or anything above u except via the edge (u, v) itself.

**Articulation point conditions:**
- Root of DFS tree: articulation iff it has ≥ 2 DFS-tree children
- Non-root u: articulation iff some child v has `low[v] >= disc[u]`

This stuff is rare in interviews; if you see it, you'll have time to look it up. Just know it exists and the time complexity.

**Practice — advanced graph (rare, senior rounds)**

| Concept | LeetCode |
|---|---|
| Critical connections in a network (bridges) | LC 1192 |
| Strongly connected components | (GFG classic) |


---
# 17. Grids as graphs — a huge interview category

## The picture
An `m x n` grid is just a graph where each cell is a vertex and each cell connects to its 4 (or 8) neighbours. No need to explicitly build an adjacency list — generate neighbours on demand.

This category is enormous in interviews because it's intuitive and visual. Number of islands, rotting oranges, shortest path in a grid, word search — all of these.

## The canonical neighbour generator
```python
for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
    nr, nc = r + dr, c + dc
    if 0 <= nr < m and 0 <= nc < n:
        # nr, nc is a valid neighbour
```

## Visited tracking
Two options:
1. Separate `visited` 2D array (clean)
2. Mark in place (faster, no extra space) — e.g., set the cell to a sentinel like `'#'` or `'0'`. Revert if you need to preserve the input.

## Patterns specific to grids
- **Single-source BFS/DFS** — flood fill, count one island
- **Multi-source BFS** — rotting oranges, distance to nearest 0
- **0-1 BFS** — when some moves are free
- **Dijkstra on grid** — when each cell has a cost (e.g., LC 1631)
- **BFS with extra state** — when you can break walls (LC 1293) or carry keys (LC 864)

In [ ]:
# 17.1 Number of islands — LC 200 — flood-fill DFS
def num_islands(grid):
    if not grid: return 0
    m, n = len(grid), len(grid[0])
    count = 0
    def dfs(r, c):
        if r < 0 or r >= m or c < 0 or c >= n or grid[r][c] != '1':
            return
        grid[r][c] = '#'                # mark visited in place
        dfs(r-1, c); dfs(r+1, c); dfs(r, c-1); dfs(r, c+1)
    for r in range(m):
        for c in range(n):
            if grid[r][c] == '1':
                count += 1
                dfs(r, c)
    return count

g = [["1","1","1","1","0"],
     ["1","1","0","1","0"],
     ["1","1","0","0","0"],
     ["0","0","0","0","0"]]
print(num_islands(g))                   # 1

g = [["1","1","0","0","0"],
     ["1","1","0","0","0"],
     ["0","0","1","0","0"],
     ["0","0","0","1","1"]]
print(num_islands(g))                   # 3


In [ ]:
# 17.2 Rotting oranges — LC 994 — multi-source BFS
def oranges_rotting(grid):
    m, n = len(grid), len(grid[0])
    q = deque()
    fresh = 0
    # Initialize: ALL rotten oranges go in the queue at the START with distance 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 2:
                q.append((r, c, 0))
            elif grid[r][c] == 1:
                fresh += 1
    if fresh == 0: return 0
    last_time = 0
    while q:
        r, c, t = q.popleft()
        last_time = max(last_time, t)
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == 1:
                grid[nr][nc] = 2        # rot it
                fresh -= 1
                q.append((nr, nc, t+1))
    return last_time if fresh == 0 else -1

print(oranges_rotting([[2,1,1],[1,1,0],[0,1,1]]))   # 4
print(oranges_rotting([[2,1,1],[0,1,1],[1,0,1]]))   # -1 (the bottom-left 1 is unreachable)


In [ ]:
# 17.3 Shortest path in binary matrix — LC 1091 — BFS with 8 directions
def shortest_path_binary_matrix(grid):
    n = len(grid)
    if grid[0][0] != 0 or grid[n-1][n-1] != 0:
        return -1
    dirs = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
    q = deque([(0, 0, 1)])              # (r, c, path_length so far)
    visited = [[False]*n for _ in range(n)]
    visited[0][0] = True
    while q:
        r, c, d = q.popleft()
        if r == n-1 and c == n-1:
            return d
        for dr, dc in dirs:
            nr, nc = r+dr, c+dc
            if 0 <= nr < n and 0 <= nc < n and not visited[nr][nc] and grid[nr][nc] == 0:
                visited[nr][nc] = True
                q.append((nr, nc, d+1))
    return -1

print(shortest_path_binary_matrix([[0,1],[1,0]]))                    # 2
print(shortest_path_binary_matrix([[0,0,0],[1,1,0],[1,1,0]]))        # 4


In [ ]:
# 17.4 01 Matrix — LC 542 — multi-source BFS from all 0s
def update_matrix(mat):
    m, n = len(mat), len(mat[0])
    INF = float('inf')
    dist = [[INF]*n for _ in range(m)]
    q = deque()
    # Seed: all 0s have distance 0
    for r in range(m):
        for c in range(n):
            if mat[r][c] == 0:
                dist[r][c] = 0
                q.append((r, c))
    while q:
        r, c = q.popleft()
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < m and 0 <= nc < n and dist[nr][nc] > dist[r][c] + 1:
                dist[nr][nc] = dist[r][c] + 1
                q.append((nr, nc))
    return dist

print(update_matrix([[0,0,0],[0,1,0],[1,1,1]]))


**Why multi-source BFS works:** if you push all sources into the queue at the start, BFS treats them as a unified "level 0." The first time any cell is reached, it's at the minimum distance to its NEAREST source — which is exactly what these problems ask.

**Practice — grids as graphs**

| Concept | LeetCode |
|---|---|
| Number of islands | LC 200 |
| Max area of island | LC 695 |
| Number of distinct islands | LC 694 |
| Flood fill | LC 733 |
| Pacific Atlantic water flow | LC 417 |
| Surrounded regions | LC 130 |
| Rotting oranges | LC 994 |
| 01 Matrix | LC 542 |
| Walls and gates | LC 286 |
| Shortest path in binary matrix | LC 1091 |
| Shortest path with obstacle elimination | LC 1293 |
| Swim in rising water | LC 778 (Dijkstra on grid) |
| Path with minimum effort | LC 1631 |
| Word search | LC 79 |
| Number of islands II (Union-Find) | LC 305 |
| Making a large island | LC 827 |


---
# 18. Pattern-recognition cheat sheet

| Problem signal | Reach for |
|---|---|
| "Shortest path / steps", unweighted | BFS |
| "Connected components / count islands / regions" | BFS, DFS, or Union-Find |
| "Detect a cycle" — undirected | DFS with parent tracking, or Union-Find |
| "Detect a cycle" — directed | DFS with recursion stack (3-color), or Kahn's |
| "Order of tasks with dependencies" | Topological sort (Kahn's or DFS) |
| "Course prerequisites / can you finish" | Topo sort + cycle check |
| "Shortest path with positive weights" | Dijkstra |
| "Shortest path with 0/1 weights" | 0-1 BFS (deque trick) |
| "Shortest path with possible negatives / detect negative cycle" | Bellman-Ford |
| "All-pairs shortest path on dense graph" | Floyd-Warshall |
| "Cheapest way to connect all nodes" | MST — Prim's (dense) or Kruskal's (sparse) |
| "Edges arrive over time, are these connected yet?" | Union-Find |
| "Spread / flood / multiple sources at the same time" | Multi-source BFS |
| "Each cell has a cost, find cheapest path" | Dijkstra on the grid |
| "Strongly connected components / collapsing graph into a DAG" | Kosaraju / Tarjan |
| "Critical edges / single point of failure" | Bridges, articulation points |

**Choosing representation:**
- Default to **adjacency list**
- Use **adjacency matrix** for: dense graphs, frequent "is there an edge?" queries, Floyd-Warshall

**Time complexity summary:**

| Algorithm | Time | Space |
|---|---|---|
| BFS / DFS | O(V + E) | O(V) |
| Topological sort | O(V + E) | O(V) |
| Dijkstra (heap) | O((V+E) log V) | O(V) |
| Dijkstra (matrix) | O(V²) | O(V²) |
| Bellman-Ford | O(V · E) | O(V) |
| Floyd-Warshall | O(V³) | O(V²) |
| Prim's (heap) | O((V+E) log V) | O(V) |
| Kruskal's | O(E log E) | O(V) |
| Union-Find op | O(α(n)) ≈ O(1) | O(V) |
| Kosaraju SCC | O(V + E) | O(V + E) |


---
# 19. Interview narration & common bugs

## 19a. The opening narration (ask before coding)
1. "Is the graph directed or undirected?"
2. "Are there weights? Can they be negative?"
3. "Is it guaranteed connected? Can there be multiple components?"
4. "Can there be self-loops or multi-edges?"
5. "What's the range of V and E? Sparse or dense?"
6. "Is the input edges, or already adjacency list, or a 2D grid?"

These six answers narrow the algorithm to one or two candidates.

## 19b. How to explain BFS vs DFS in a sentence
> "BFS explores level by level using a queue, which makes it perfect for unweighted shortest paths. DFS goes deep before backtracking using a stack (often implicit via recursion), which makes it perfect for cycle detection, topological sort, and exhaustive path exploration."

## 19c. How to explain why Dijkstra fails on negatives
> "Dijkstra is greedy — once it finalizes a vertex's distance, it never reconsiders. With a negative edge, a longer-looking path could become cheaper after the fact, breaking that greedy choice. Bellman-Ford handles this by relaxing every edge V-1 times instead of greedily finalizing."

## 19d. How to explain Union-Find's near-O(1) complexity
> "Union-Find with path compression and union by rank has amortized O(α(n)) per operation, where α is the inverse Ackermann function. For any practical n — even up to 10⁸⁰ — α(n) is at most 4. So it's effectively constant time."

## 19e. Common bugs

1. **Using `list.pop(0)` as a BFS queue.** O(n) per pop makes the algorithm O(V²). Use `collections.deque` and `popleft()`.

2. **Marking visited on dequeue, not enqueue.** Causes duplicates in the queue and re-processing.

3. **Undirected cycle detection without parent tracking.** Every edge looks like a cycle.

4. **Directed cycle detection with only `visited[]`.** You need `recursion_stack[]` (or 3 colors) to distinguish back edges from cross/forward edges.

5. **Dijkstra without "stale heap entry" check.** Add `if d > dist[u]: continue` after popping.

6. **Using Dijkstra on negative edges.** Just wrong. Switch to Bellman-Ford.

7. **Forgetting that Dijkstra is single-source.** If you need all-pairs, run V Dijkstras or use Floyd-Warshall.

8. **Floyd-Warshall with wrong loop order.** `k` must be the OUTERMOST loop. Always.

9. **Topological sort on a graph with a cycle.** Returns garbage. Always check that `len(order) == V` at the end.

10. **Forgetting to handle disconnected graphs.** Wrap your BFS/DFS in an outer loop that walks all vertices and only launches a traversal from unvisited ones.

11. **Recursion depth overflow on long chains.** Python's default limit is ~1000. For graphs that could have a chain of 10000 nodes, use iterative DFS or `sys.setrecursionlimit(10**6)`.

12. **Adjacency matrix when V is huge.** V=10⁵ → V² = 10¹⁰ cells, way too much. Always check the input size before choosing matrix.

13. **Confusing `find()` with `parent[]` in Union-Find.** `parent[x]` is just the next node up; `find(x)` walks to the root. Always call `find` before comparing.

14. **Not unioning by rank/size.** Without it, the worst-case Union-Find degrades to O(n) per op.

15. **Building adjacency list twice for undirected graphs.** `adj[u].append(v); adj[v].append(u)` once. People sometimes loop both directions of edges, which double-adds.

16. **MST on a disconnected graph.** Kruskal's gracefully detects this (you can't pick V-1 edges); make sure to return an error / -1, not a partial answer.

17. **Grid problems: missing the "starting cell is blocked" edge case.** Always check `grid[0][0]` before BFS.

18. **Using a `set` for visited in performance-critical inner loops.** A list of bools (or a 2D matrix for grids) is much faster.
